In [2]:
# NOTEBOOK 1 — UIDAI CAPACITY PRESSURE SCORE (CPS)


import pandas as pd
import numpy as np

print("=" * 70)
print("UIDAI — Caapacity pressure score")
print("Notebook 1: Core Metric Construction")
print("=" * 70)

print("\nLibraries loaded:")
print("pandas")
print("numpy")


UIDAI — Caapacity pressure score
Notebook 1: Core Metric Construction

Libraries loaded:
pandas
numpy


In [3]:

demographic_master = pd.read_csv("demographic_master.csv")
enrolment_master = pd.read_csv("enrolment_master.csv")
biometric_master = pd.read_csv("biometric_master.csv")

print("Master datasets loaded.")

print("Record counts:")
print("Demographic:", len(demographic_master))
print("Enrolment  :", len(enrolment_master))
print("Biometric  :", len(biometric_master))

print("\nColumn verification:")
print("Demographic:", demographic_master.columns.tolist())
print("Enrolment  :", enrolment_master.columns.tolist())
print("Biometric  :", biometric_master.columns.tolist())


Master datasets loaded.
Record counts:
Demographic: 2071700
Enrolment  : 1006029
Biometric  : 1861108

Column verification:
Demographic: ['date', 'state', 'district', 'pincode', 'demo_age_5_17', 'demo_age_17_']
Enrolment  : ['date', 'state', 'district', 'pincode', 'age_0_5', 'age_5_17', 'age_18_greater']
Biometric  : ['date', 'state', 'district', 'pincode', 'bio_age_5_17', 'bio_age_17_']


In [7]:
demographic_master['date'] = pd.to_datetime(
    demographic_master['date'], format='%d-%m-%Y'
)
enrolment_master['date'] = pd.to_datetime(
    enrolment_master['date'], format='%d-%m-%Y'
)
biometric_master['date'] = pd.to_datetime(
    biometric_master['date'], format='%d-%m-%Y'
)

# Create month key (YYYY-MM)
demographic_master['month'] = demographic_master['date'].dt.to_period('M')
enrolment_master['month'] = enrolment_master['date'].dt.to_period('M')
biometric_master['month'] = biometric_master['date'].dt.to_period('M')

print("Date parsing and monthly period alignment completed.")

print("\nDate ranges:")
print("Demographic:", demographic_master['date'].min(), "to", demographic_master['date'].max())
print("Enrolment  :", enrolment_master['date'].min(), "to", enrolment_master['date'].max())
print("Biometric  :", biometric_master['date'].min(), "to", biometric_master['date'].max())

print("\nUnique months available:")
print("Demographic:", demographic_master['month'].nunique())
print("Enrolment  :", enrolment_master['month'].nunique())
print("Biometric  :", biometric_master['month'].nunique())


print("\nSample month values:")
print(demographic_master[['date', 'month']].head())


Date parsing and monthly period alignment completed.

Date ranges:
Demographic: 2025-03-01 00:00:00 to 2025-12-29 00:00:00
Enrolment  : 2025-03-02 00:00:00 to 2025-12-31 00:00:00
Biometric  : 2025-03-01 00:00:00 to 2025-12-29 00:00:00

Unique months available:
Demographic: 9
Enrolment  : 9
Biometric  : 9

Sample month values:
        date    month
0 2025-03-01  2025-03
1 2025-03-01  2025-03
2 2025-03-01  2025-03
3 2025-03-01  2025-03
4 2025-03-01  2025-03


In [8]:
print("\nstate validation in progress")

print("Demographic numeric states:",
      demographic_master['state'].str.match(r'^\d+$', na=False).sum())

print("Enrolment numeric states:",
      enrolment_master['state'].str.match(r'^\d+$', na=False).sum())

print("Biometric numeric states:",
      biometric_master['state'].str.match(r'^\d+$', na=False).sum())



demographic_clean = demographic_master[
    ~demographic_master['state'].str.match(r'^\d+$', na=False)
].copy()

enrolment_clean = enrolment_master[
    ~enrolment_master['state'].str.match(r'^\d+$', na=False)
].copy()

biometric_clean = biometric_master[
    ~biometric_master['state'].str.match(r'^\d+$', na=False)
].copy()



def standardize_state(name):
    if pd.isna(name):
        return None
    if name in ['Andaman and Nicobar Islands', 'Andaman & Nicobar Islands']:
        return 'Andaman & Nicobar Islands'
    return name

demographic_clean['state'] = demographic_clean['state'].apply(standardize_state)
enrolment_clean['state'] = enrolment_clean['state'].apply(standardize_state)
biometric_clean['state'] = biometric_clean['state'].apply(standardize_state)



print("\nState validation complete")

print("\nRecords removed due to malformed states:")
print("Demographic:", len(demographic_master) - len(demographic_clean))
print("Enrolment  :", len(enrolment_master) - len(enrolment_clean))
print("Biometric  :", len(biometric_master) - len(biometric_clean))

print("\nUnique states after cleaning:")
print("Demographic:", demographic_clean['state'].nunique())
print("Enrolment  :", enrolment_clean['state'].nunique())
print("Biometric  :", biometric_clean['state'].nunique())

print("\nTop 10 states (by record count, demographic):")
print(demographic_clean['state'].value_counts().head(10))



state validation in progress
Demographic numeric states: 2
Enrolment numeric states: 22
Biometric numeric states: 0

State validation complete

Records removed due to malformed states:
Demographic: 2
Enrolment  : 22
Biometric  : 0

Unique states after cleaning:
Demographic: 63
Enrolment  : 53
Biometric  : 56

Top 10 states (by record count, demographic):
state
Andhra Pradesh    207687
Tamil Nadu        196857
West Bengal       168623
Uttar Pradesh     167889
Maharashtra       162242
Karnataka         153957
Kerala            105515
Bihar              97621
Gujarat            96399
Odisha             92143
Name: count, dtype: int64


In [ ]:
print("\nAggregating enrolment and demographic data at state-month level.")


state_month_enrol = (
    enrolment_clean
    .groupby(['state', 'month'])[['age_0_5', 'age_5_17', 'age_18_greater']]
    .sum()
    .reset_index()
)

state_month_demo = (
    demographic_clean
    .groupby(['state', 'month'])[['demo_age_5_17', 'demo_age_17_']]
    .sum()
    .reset_index()
)

print("\nAggregations complete")

print("Enrolment aggregation shape:", state_month_enrol.shape)
print("Demographic aggregation shape:", state_month_demo.shape)

print("\nSample enrolment aggregation:")
print(state_month_enrol.head())

print("\nSample demographic aggregation:")
print(state_month_demo.head())



Aggregating enrolment and demographic data at state-month level.

Aggregations complete
Enrolment aggregation shape: (315, 5)
Demographic aggregation shape: (351, 4)

Sample enrolment aggregation:
                       state    month  age_0_5  age_5_17  age_18_greater
0  Andaman & Nicobar Islands  2025-09      172        16               0
1  Andaman & Nicobar Islands  2025-10       74         8               0
2  Andaman & Nicobar Islands  2025-11      109         3               0
3  Andaman & Nicobar Islands  2025-12      124         5               0
4             Andhra Pradesh  2025-03       43        44              29

Sample demographic aggregation:
                       state    month  demo_age_5_17  demo_age_17_
0  Andaman & Nicobar Islands  2025-03            126          1212
1  Andaman & Nicobar Islands  2025-07             40           398
2  Andaman & Nicobar Islands  2025-09             77           961
3  Andaman & Nicobar Islands  2025-10             75           

In [14]:

print("\nMerging enrolment and demographic agregates.")

state_month_combined = (
    state_month_enrol
    .merge(
        state_month_demo,
        on=['state', 'month'],
        how='inner'
    )
)


print("Combined table shape (before threshold):", state_month_combined.shape)


MIN_ADULT_ENROLMENTS = 100

state_month_combined = state_month_combined[
    state_month_combined['age_18_greater'] >= MIN_ADULT_ENROLMENTS
].copy()


print(f"Threshold: age_18_greater ≥ {MIN_ADULT_ENROLMENTS}")
print("Combined table shape (after threshold):", state_month_combined.shape)


print(
    state_month_combined[
        ['state', 'month', 'age_18_greater', 'demo_age_17_']
    ].head()
)



Merging enrolment and demographic agregates.
Combined table shape (before threshold): (286, 7)
Threshold: age_18_greater ≥ 100
Combined table shape (after threshold): (141, 7)
            state    month  age_18_greater  demo_age_17_
5  Andhra Pradesh  2025-06             213        109494
6  Andhra Pradesh  2025-07             111         74539
7  Andhra Pradesh  2025-09             379        243849
8  Andhra Pradesh  2025-10             140        194941
9  Andhra Pradesh  2025-11             385        328191


In [15]:

print("\nComputing update-to-enrolment ratios")

# Adult update ratio (primary signal)
state_month_combined['adult_update_ratio'] = (
    state_month_combined['demo_age_17_'] /
    state_month_combined['age_18_greater']
)

# Child update ratio (secondary signal)
state_month_combined['child_update_ratio'] = (
    state_month_combined['demo_age_5_17'] /
    state_month_combined['age_5_17'].replace(0, np.nan)
)

# Adult-to-child skew (pressure imbalance indicator)
state_month_combined['adult_child_skew'] = (
    state_month_combined['adult_update_ratio'] /
    (state_month_combined['child_update_ratio'] + 0.01)
)

print("\nSample ratio values:")
print(
    state_month_combined[
        ['state', 'month',
         'adult_update_ratio',
         'child_update_ratio',
         'adult_child_skew']
    ].head()
)

print("\nBasic sanity checks:")
print("Adult update ratio — min:", round(state_month_combined['adult_update_ratio'].min(), 2),
      "max:", round(state_month_combined['adult_update_ratio'].max(), 2))

print("Number of state-month rows:", len(state_month_combined))



Computing update-to-enrolment ratios

Sample ratio values:
            state    month  adult_update_ratio  child_update_ratio  \
5  Andhra Pradesh  2025-06          514.056338           52.711316   
6  Andhra Pradesh  2025-07          671.522523           34.817204   
7  Andhra Pradesh  2025-09          643.401055            5.043875   
8  Andhra Pradesh  2025-10         1392.435714           14.335818   
9  Andhra Pradesh  2025-11          852.444156           21.912541   

   adult_child_skew  
5          9.750446  
6         19.281551  
7        127.308454  
8         97.062131  
9         38.884369  

Basic sanity checks:
Adult update ratio — min: 0.52 max: 3390.9
Number of state-month rows: 141


In [16]:
print("\nAggregating metrics at state level")

state_stats = (
    state_month_combined
    .groupby('state')
    .agg(
        avg_update_ratio=('adult_update_ratio', 'mean'),
        update_volatility=('adult_update_ratio', 'std'),
        avg_adult_child_skew=('adult_child_skew', 'mean'),
        total_adult_enrolments=('age_18_greater', 'sum'),
        total_adult_updates=('demo_age_17_', 'sum')
    )
    .reset_index()
)

# Replace NaN volatility (states with single observation)
state_stats['update_volatility'] = state_stats['update_volatility'].fillna(0)
print("States analyzed:", len(state_stats))

print("\nSample state-level metrics:")
print(
    state_stats[
        ['state',
         'avg_update_ratio',
         'update_volatility',
         'avg_adult_child_skew']
    ].head()
)

print("\nSanity checks:")
print("Avg update ratio — min:",
      round(state_stats['avg_update_ratio'].min(), 2),
      "max:",
      round(state_stats['avg_update_ratio'].max(), 2))

print("Volatility — min:",
      round(state_stats['update_volatility'].min(), 2),
      "max:",
      round(state_stats['update_volatility'].max(), 2))



Aggregating metrics at state level
States analyzed: 23

Sample state-level metrics:
            state  avg_update_ratio  update_volatility  avg_adult_child_skew
0  Andhra Pradesh        948.804362         450.416129             54.801646
1           Assam         81.142053         101.449587             41.124326
2           Bihar        727.041722         803.829220            405.530757
3    Chhattisgarh        898.022249         612.459440            122.109690
4           Delhi        414.847959         304.800686             65.611118

Sanity checks:
Avg update ratio — min: 13.24 max: 1300.01
Volatility — min: 0.0 max: 1299.05


In [ ]:
#  Normalize CPS Components to 0–100 Scale

print("\nNormalizing CPS components to 0–100 scale.")

def normalize_0_100(series):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series([50] * len(series), index=series.index)
    return ((series - min_val) / (max_val - min_val)) * 100

state_stats['norm_update_intensity'] = normalize_0_100(
    state_stats['avg_update_ratio']
)

state_stats['norm_volatility'] = normalize_0_100(
    state_stats['update_volatility']
)

state_stats['norm_adult_child_skew'] = normalize_0_100(
    state_stats['avg_adult_child_skew']
)

print("\nSample normalized values:")
print(
    state_stats[
        ['state',
         'norm_update_intensity',
         'norm_volatility',
         'norm_adult_child_skew']
    ].head()
)

print("\nRange checks:")
print("Update intensity:", 
      
      round(state_stats['norm_update_intensity'].min(), 1), "→",
      round(state_stats['norm_update_intensity'].max(), 1))

print("Volatility:", 
      round(state_stats['norm_volatility'].min(), 1), "→",
      round(state_stats['norm_volatility'].max(), 1))

print("Adult-child skew:", 
      round(state_stats['norm_adult_child_skew'].min(), 1), "→",
      round(state_stats['norm_adult_child_skew'].max(), 1))



Normalizing CPS components to 0–100 scale.

Sample normalized values:
            state  norm_update_intensity  norm_volatility  \
0  Andhra Pradesh              72.706310        34.672606   
1           Assam               5.276628         7.809493   
2           Bihar              55.472204        61.878011   
3    Chhattisgarh              68.759819        47.146547   
4           Delhi              31.210315        23.463268   

   norm_adult_child_skew  
0               8.474779  
1               6.004167  
2              71.828963  
3              20.633012  
4              10.427355  

Range checks:
Update intensity: 0.0 → 100.0
Volatility: 0.0 → 100.0
Adult-child skew: 0.0 → 100.0


In [ ]:
# Capacity Pressure Score (CPS) Construction

print("\nConstructing Capacity Pressure Score (CPS).")

state_stats['CPS'] = (
    0.40 * state_stats['norm_update_intensity'] +
    0.30 * state_stats['norm_volatility'] +
    0.30 * state_stats['norm_adult_child_skew']
)



print("\nSample CPS values:")
print(
    state_stats[['state', 'CPS']]
    .sort_values('CPS', ascending=False)
    .head()
)

print("\nCPS distribution summary:")
print(state_stats['CPS'].describe())



Constructing Capacity Pressure Score (CPS).

Sample CPS values:
          state        CPS
22  West Bengal  80.914509
19   Tamil Nadu  76.145158
8     Jharkhand  68.661253
2         Bihar  62.300973
20    Telangana  60.688250

CPS distribution summary:
count    23.000000
mean     32.469859
std      24.751732
min       0.432670
25%      15.061389
50%      24.594498
75%      51.122108
max      80.914509
Name: CPS, dtype: float64


In [19]:
#  CPS Classification & Persistence

print("\nClassifying states based on CPS thresholds.")

def classify_cps(score):
    if score >= 70:
        return "Red"
    elif score >= 40:
        return "Yellow"
    else:
        return "Green"

state_stats['CPS_Category'] = state_stats['CPS'].apply(classify_cps)

# Sort by CPS descending for clarity
cps_final = state_stats.sort_values('CPS', ascending=False).reset_index(drop=True)

print("\nCPS Category distribution:")
print(cps_final['CPS_Category'].value_counts())

print("\nFinal CPS table (top 10 states):")
print(
    cps_final[['state', 'CPS', 'CPS_Category']]
    .head(10)
)

# Save final CPS output
cps_final.to_csv("state_cps_final.csv", index=False)

print("\nCPS distribution summary:")
print(state_stats['CPS'].describe())



Classifying states based on CPS thresholds.

CPS Category distribution:
CPS_Category
Green     15
Yellow     6
Red        2
Name: count, dtype: int64

Final CPS table (top 10 states):
            state        CPS CPS_Category
0     West Bengal  80.914509          Red
1      Tamil Nadu  76.145158          Red
2       Jharkhand  68.661253       Yellow
3           Bihar  62.300973       Yellow
4       Telangana  60.688250       Yellow
5   Uttar Pradesh  54.406422       Yellow
6    Chhattisgarh  47.837795       Yellow
7  Andhra Pradesh  42.026740       Yellow
8          Odisha  34.905790        Green
9     Maharashtra  31.323882        Green

CPS distribution summary:
count    23.000000
mean     32.469859
std      24.751732
min       0.432670
25%      15.061389
50%      24.594498
75%      51.122108
max      80.914509
Name: CPS, dtype: float64


In [20]:

state_stats.to_csv("state_cps_final.csv", index=False)

print("Output written to file: state_cps_final.csv")
print("Notebook 1 completed.")



Output written to file: state_cps_final.csv
Notebook 1 completed.
